In [1]:
#IMPORTS
from pynq import Overlay, allocate, PL, MMIO
from pynq.lib.video import VideoMode
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
from enum import Enum
import time

PL.reset()
overlay = Overlay("/home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/VivadoHW.bit")
overlay.download()

In [2]:
#SETUP AND DEFINES
DEFAULT_WIDTH, DEFAULT_HEIGHT = 1280, 720
tpg = overlay.v_tpg_0
vdma = overlay.axi_vdma_0
buttons = overlay.axi_gpios.axi_gpio_0
imageResolution = overlay.axi_gpios.axi_gpio_1
image_widget = widgets.Image(format='jpeg', width=DEFAULT_WIDTH, height=DEFAULT_HEIGHT)
stream_switch = overlay.orb_pipeline.axis_switch_0

class VideoSource(Enum):
    HDMI = 0
    TPG_COLOR_BAR_BOX = 1
    TPG_GRID_BOX = 2


selected_source = None

In [5]:
#MAIN
def setupVideoSource(new_src: VideoSource):
    global selected_source 
    if selected_source == new_src:
        return
    
    was_running = vdma.readchannel.running
    if was_running:
        print("vdma was running restarting it to setup new video source")
        vdma.readchannel.stop()

    selected_source = new_src

    if(selected_source in [VideoSource.HDMI,VideoSource.TPG_COLOR_BAR_BOX,VideoSource.TPG_GRID_BOX]):
        imageResolution.channel1.write(DEFAULT_HEIGHT, 0xFFFFFFFF)
        imageResolution.channel2.write(DEFAULT_WIDTH, 0xFFFFFFFF)
    
    tpg.write(0x0000, 0x0)
    tpg.write(0x0010, DEFAULT_HEIGHT)
    tpg.write(0x0018, DEFAULT_WIDTH)

    if selected_source == VideoSource.TPG_COLOR_BAR_BOX:
        tpg.write(0x0020, 9); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)
        tpg.write(0x0000, 0x81)
    elif selected_source == VideoSource.TPG_GRID_BOX:
        tpg.write(0x0020, 11); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)
        tpg.write(0x0000, 0x81)

    stream_switch.write(0x040, int(selected_source != VideoSource.HDMI))
    stream_switch.write(0x000, 0x2)
    
    if was_running:
        vdma.readchannel.start()

vdma.readchannel.mode = VideoMode(DEFAULT_WIDTH, DEFAULT_HEIGHT, 8)
setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
vdma.readchannel.start()

display(image_widget)

print("Streaming via VDMA interrupts...")
old_button_value = buttons.read(0)

try:
    while True:
        #READ AND DISPLAY FRAME

        frame = vdma.readchannel.readframe()
        _, jpeg = cv2.imencode('.jpg', frame)
        image_widget.value = jpeg.tobytes()
        #DETECT BUTTON AND SWITCH VIDEO SOURCE
        button_value = buttons.read(0)
        if button_value != old_button_value and button_value > 0:
            if button_value & 0x1:
                setupVideoSource(VideoSource.HDMI)
            elif button_value & 0x2:
                setupVideoSource(VideoSource.TPG_COLOR_BAR_BOX)
            elif button_value & 0x4:
                setupVideoSource(VideoSource.TPG_GRID_BOX)
        old_button_value = button_value
        

except KeyboardInterrupt:
    print("\nStopping stream...")
finally:
    tpg.write(0x00, 0x0)
    vdma.readchannel.stop()
    print("Stream stopped cleanly.")

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

Streaming via VDMA interrupts...
vdma was running restarting it to setup new video source
vdma was running restarting it to setup new video source

Stopping stream...
Stream stopped cleanly.
